# Gekoppelte Pendel


In [ ]:
# Import all the numerics and plotting packages
# pylab inline
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from IPython.core.pylabtools import figsize, getfigs
from numpy import *
from IPython.display import Video, display

In [ ]:
from pathlib import Path

# Verzeichnis für exportierte Abbildungen und Videos
notebook_name = "CoupledPendulum"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "01_mechanik" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)

# Abbildungsstil für ein A4-Skript
latex_textwidth_in = 6.3
single_panel_width = 0.5 * latex_textwidth_in
single_panel_square = (single_panel_width, single_panel_width)
single_panel_wide = (single_panel_width, 0.68 * single_panel_width)
full_width_three_panel = (latex_textwidth_in, 0.36 * latex_textwidth_in)
animation_display_width = 360

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "lines.linewidth": 1.2,
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print(f"Output directory: {output_dir}")

In [ ]:
dim=2 #fest

# Definitioon der Struktur (Hier Haus vom Nikolaus aus VL)

num_points=3 # Anzahl Massenpunkte
#Massen der Pubkte (Einheit ist Masse 1)
Masses=np.array([1,1,1])
#Ruhelagen (Rs[j,0],Rs[j,1]) ist Rghelage von Punkt j
Rs = np.zeros((num_points,dim))
Rs[0, :] = np.array([0,2])
Rs[1, :] = np.array([1,3])
Rs[2, :] = np.array([2,2])

#Liste von Punkten, an denen federn fest bestigt sind
num_fpoints=2
Fs = np.zeros((num_fpoints,dim))
Fs[0, :] = np.array([0,0])
Fs[1, :] = np.array([2,0])

# Verbindungen zwichen Massenpunkten (zähle jede verbindung nur einmal!)
# K_RR[i,j] ist federkonstante zwischen Masse i und j
# K_RF[i,j] ist federkonstante zwischen Masse i und Fixpunkt j
K_RR = np.zeros((num_points,num_points))
K_RR[0,2]=1
K_RR[0,1]=1
K_RR[1,2]=1
K_RF = np.zeros((num_points,num_fpoints))
K_RF[0,0]=1
K_RF[0,1]=1
K_RF[2,0]=1
K_RF[2,1]=1

#Anzahl Freiheitsgrade
n=2*num_points 

In [ ]:
#Bestimmung der Dynamischen Matrix (siehe Vorlesung)

# Dynamische Matrix für eine Feder zwischen zwei Massen i und j
def D_mass_to_mass(i,j):
    # Richtungsvektror längs Feder
    R=Rs[i,:]-Rs[j,:]
    # Einheitsvektor
    R_hat=R/np.sqrt(R.dot(R))
    # vector R_hat*delR = C[i]*q[i]
    C=np.zeros(n)
    C[2*i]=R_hat[0]
    C[2*i+1]=R_hat[1]
    C[2*j]=-R_hat[0]
    C[2*j+1]=-R_hat[1]    
    Dtmp = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            Dtmp[i, j] = C[i]*C[j]
    return Dtmp

# Dynamische Matrix für eine Feder zwischen von einem Fixpunkt R0 zur MAsse i
def D_mass_to_fixpoint(i,j):
    # Richtungsvektror längs Feder
    R=Rs[i,:]-Fs[j,:]
    # Einheitsvektor
    norm=np.sqrt(R.dot(R))
    R_hat=R/norm
    #print("i,j=",i,j,"hat_R=",R_hat)
    # vector R_hat*delR = C[i]*q[i]
    C=np.zeros(n)
    C[2*i]=R_hat[0]
    C[2*i+1]=R_hat[1]
    Dtmp = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            Dtmp[i, j] = C[i]*C[j]
    return Dtmp

# Konstruktion der dyanmischen Matrix
D = np.zeros((n, n))

for i in range(num_points):
    for j in range(i):
        D += K_RR[j,i]*D_mass_to_mass(j,i)

for i in range(num_points):
    for j in range(num_fpoints):
        D += K_RF[i,j]*D_mass_to_fixpoint(i,j)



In [ ]:
# Massenmatrix
M=np.zeros(n)
for j in range(num_points):
    M[2*j]=Masses[j]
    M[2*j+1]=Masses[j]

M_sqrt_inv = np.diag(1 / np.sqrt(M))
M_sqrt = np.diag(np.sqrt(M))


In [ ]:
# Entkopplung der Bewegungsgleichungen:

#Skalierte D-Matrix
Dtilde = M_sqrt_inv @ D @ M_sqrt_inv

#Diagonalisierung:
eigenvalues, eigenvectors = np.linalg.eigh(Dtilde)

# W is the matrix of eigenvectors
W = eigenvectors

# Diagonal matrix of eigenvalues 

Omegas = np.sqrt(eigenvalues)+1e-10*np.ones(n)  
# die Korrektur 1e-10*np.ones(n)  ist nur ein billiger Trick, damit 
# für Schwingungen mit mit Omega=0 der Faktor sin(omega*t)/Omega nicht divergiert

print(Omegas)

In [ ]:
#Lösung der Bewegung für gegebenen Anfangsgsbedingungen von p und q:

# FALL 1: beliebige AB: 
p_0=np.zeros(n)
q_0=np.zeros(n)
q_0[0]=0.3

#FALL2 : AB lägngs Eigenschwingung alpha
#alpha=0
#p_0=np.zeros(n)
#As=M_sqrt_inv@W
#q_0=As[:,alpha]*0.2

#Anfangsbedingunegn für Q und P:
qq_0=W.T@M_sqrt@q_0
pp_0=W.T@M_sqrt_inv@p_0

nt=100 #Anzahl Zeitschritte
h=0.3  #Zeitschritt für darstellung der Lösung
ts=h*np.arange(nt)
q_t=np.zeros((nt,n))
for idx,t in enumerate(ts):
    Qt=qq_0*np.cos(t*Omegas)+pp_0*np.sin(t*Omegas)/Omegas
    q_t[idx,:]=M_sqrt_inv@W@Qt

    
#plotting:   

fig , ax = plt.subplots(figsize=single_panel_square)
for j in range(n):
    ax.plot(ts, q_t[:,j],linestyle='-', label=f'q_{j:d}')


ax.set_xlabel("time")
ax.set_ylabel("x")
ax.legend()
plt.show()
fig.savefig(output_dir / "nikolaus_oscillation_01.pdf", bbox_inches="tight")

In [ ]:
from matplotlib.animation import FuncAnimation

#Koordinaten für alle massen:
r_t = np.zeros((nt, num_points, 2))

# Compute r_t
for t in range(nt):
    for j in range(num_points):
        r_t[t, j, 0] = Rs[j, 0] + q_t[t, 2 * j]      # x-coordinate
        r_t[t, j, 1] = Rs[j, 1] + q_t[t, 2 * j + 1]  # y-coordinate

# Initialize plot
fig, ax = plt.subplots()
ax.set_xlim(-0.5, 2.5)  
ax.set_ylim(-0.5, 3.5)  

ax.set_aspect('equal')
circles = [plt.Circle((0, 0), 0.1, color='b', fill=True) for _ in range(num_points)]
for circle in circles:
    ax.add_patch(circle)

lines = []
for a in range(num_points):
    for b in range(a + 1, num_points):  # Only create lines for a < b
        if K_RR[a, b] != 0:
            line, = ax.plot([], [], color='gray', lw=1, zorder=1)
            lines.append((a, b, line))
flines = []
for a in range(num_points):
    for b in range(num_fpoints):  # Only create lines for a < b
        if K_RF[a, b] != 0:
            line, = ax.plot([], [], color='gray', lw=1, zorder=1)
            flines.append((a, b, line))

# Update function for animation
def update(frame):
    # Update circle positions
    for j, circle in enumerate(circles):
        circle.center = (r_t[frame, j, 0], r_t[frame, j, 1])

    # Update line positions based on current frame
    for a, b, line in lines:
        line.set_data([r_t[frame, a, 0], r_t[frame, b, 0]], 
                      [r_t[frame, a, 1], r_t[frame, b, 1]])
    for a, b, linef in flines:
        linef.set_data([r_t[frame, a, 0], Fs[b, 0]], 
                      [r_t[frame, a, 1], Fs[b, 1]])

    return circles + [line[2] for line in lines] + [linef[2] for linef in flines]    


# Create animation
ani = FuncAnimation(fig, update, frames=nt, blit=True, repeat=True)
#plt.show()
video_name = output_dir / "nikolaus_animation.mp4"
ani.save(str(video_name), writer="ffmpeg", fps=25)
plt.close(fig)
display(Video(
    str(video_name), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))

